# Step 8 — Exploratory structural explanations of resilience

This notebook is numbered `09` to preserve established output keys and manuscript references.
It uses the early/full AUC produced by the new Section 6.7, and the original validated null
ensembles. One sampled graph is one observation. The complete 378-test family per null type,
shared network-row permutations, BY adjustment and pointwise bootstrap intervals are retained.

The ancillary three-predictor CV model and descriptive per-level correlation catalog are no
longer part of the active workflow; archived original notebook/results preserve them.
No causal claim is made, and the observed networks lie outside some ensemble predictor ranges.


In [2]:
from supplementary_helpers import *
begin('09')
PREDICTORS=['Modularity','Average clustering','Transitivity','Maximum coreness','Mean coreness','Innermost-core fraction','Degree assortativity']
NPERM=19999
NBOOT=1000
mod=read('04_null_modularity_by_network.csv').set_index(['Airline','Model','Network'])
observed_mod=read('03_louvain_binary_summary.csv').set_index('Airline')
stored_structure=read('04_null_structural_measures_and_intact_baselines.csv').set_index(['Airline','Model','Network'])
observed_clustering=read('02_clustering_vs_transitivity.csv').set_index('Airline')
observed_additional=read('02_additional_structure.csv').set_index('Airline')
observed_core=read('03_kcore_summary.csv').set_index('Airline')

## Structural measurements and provenance
Modularity is reused for all 900 nulls and all observed graphs. As a consistency
audit, the same 100-seed procedure is replayed for the observed graphs and the
first graph in each ensemble. Initial-ensemble clustering and assortativity are
reused; only missing expansion measurements and core summaries are calculated.

In [3]:
struct_rows=[]
for a in AIRLINES:
    for model in MODELS:
        ids,nets=networks(a,model)
        for index,(net,s,t) in enumerate(nets):
            g=graph_undirected(ids,s,t);core=nx.core_number(g);cv=np.array(list(core.values()))
            q=float(observed_mod.loc[a,'Best modularity'] if model=='observed' else mod.loc[(a,model,net),'Best modularity'])
            if model=='observed':
                values=[observed_clustering.loc[a,'Average local clustering'],observed_clustering.loc[a,'Transitivity'],observed_additional.loc[a,'Degree assortativity']]
                source='Observed tables 02; cores recomputed and checked; modularity reused from 03'
                check(f'{a}: observed core summaries',cv.max()==observed_core.loc[a,'Maximum core number'] and np.isclose(cv.mean(),observed_core.loc[a,'Mean core number'],atol=1e-12,rtol=0) and np.isclose((cv==cv.max()).mean(),observed_core.loc[a,'Innermost core share (%)']/100,atol=1e-12,rtol=0))
            elif (a,model,net) in stored_structure.index:
                old=stored_structure.loc[(a,model,net)]
                values=[old['Average local clustering'],old.Transitivity,old['Undirected degree assortativity']]
                source='Structure reused from 04; cores newly computed; modularity reused from 04'
            else:
                values=[nx.average_clustering(g),nx.transitivity(g),nx.degree_assortativity_coefficient(g)]
                source='Expansion structure and cores newly computed; modularity reused from 04'
            if index==0:
                check(f'{a}/{model}: structural measurement audit',np.allclose(values,[nx.average_clustering(g),nx.transitivity(g),nx.degree_assortativity_coefficient(g)],atol=1e-12,rtol=0))
                best=-np.inf
                for seed in range(100):
                    communities=nx.community.louvain_communities(g,weight=None,resolution=1.0,seed=seed)
                    value=nx.community.modularity(g,communities,weight=None,resolution=1.0)
                    if value>best+1e-12:best=value
                check(f'{a}/{model}: 100-seed modularity budget audit',np.isclose(q,best,atol=1e-12,rtol=0),f'saved={q}, replay={best}')
            struct_rows.append(dict(zip(KEYS,[a,model,net]))|dict(zip(PREDICTORS,[q,values[0],values[1],cv.max(),cv.mean(),(cv==cv.max()).mean(),values[2]]))|{'Measurement source':source,'Undirected edges':g.number_of_edges(),'Reciprocity':float(np.isin(t*len(ids)+s,s*len(ids)+t).mean())})
        print(a,model,'structure complete',flush=True)
structure=pd.DataFrame(struct_rows)
check('903 unique structural rows',len(structure)==903 and not structure.duplicated(KEYS).any())
check('All structural measures finite',np.isfinite(structure[PREDICTORS].to_numpy()).all())
save(structure,'09_network_structure.csv')
areas=pd.read_csv(TABLES/'08_auc.csv')
dataset=areas.merge(structure,on=KEYS,validate='many_to_one')
check('AUC/structure join complete',len(dataset)==903*3*3*2 and dataset[PREDICTORS].notna().all().all())
save(dataset,'09_explanatory_dataset_long.csv')
positions=[]
for a in AIRLINES:
    for model in ['reciprocal','directed']:
        null=structure[(structure.Airline==a)&(structure['Network set']==model)]
        obs=structure[(structure.Airline==a)&(structure['Network set']=='observed')].iloc[0]
        for p in PREDICTORS:
            v=null[p].to_numpy();value=obs[p]
            positions.append({'Airline':a,'Null model':model,'Predictor':p,'Observed':value,'Null min':v.min(),'Null max':v.max(),'Null mean':v.mean(),'Outside sampled range':value<v.min()-1e-12 or value>v.max()+1e-12})
save(pd.DataFrame(positions),'09_observed_predictor_support.csv')

AA observed structure complete
AA reciprocal structure complete
AA directed structure complete
DL observed structure complete
DL reciprocal structure complete
DL directed structure complete
WN observed structure complete
WN reciprocal structure complete
WN directed structure complete


## Variation, dependence and multiplicity-controlled structural associations

The complete originally defined AUC comparison family (378 tests per null) is preserved,
including its seed, 19,999 shared permutations, and 1,000 pointwise bootstrap resamples.
Descriptive level-wise correlations and the small ancillary regression model were removed
from this execution path because they do not contribute a distinct corrected finding.
The original analyses and outputs remain in `Archive` for scientific traceability.


In [4]:
var_rows,assoc_rows,red_rows=[],[],[]
for ai,a in enumerate(AIRLINES):
    for mi,model in enumerate(['reciprocal','directed']):
        sg=structure[(structure.Airline==a)&(structure['Network set']==model)].sort_values('Network').set_index('Network')
        ag=areas[(areas.Airline==a)&(areas['Network set']==model)]
        wide=ag.pivot(index='Network',columns=['Strategy','Metric','Window'],values='Normalized AUC').sort_index()
        check(f'{a}/{model}: network-level alignment',sg.index.equals(wide.index))
        xstats={p:variability(sg[p]) for p in PREDICTORS};ystats={c:variability(wide[c]) for c in wide}
        for p,v in xstats.items():var_rows.append({'Airline':a,'Null model':model,'Kind':'predictor','Variable':p}|v)
        for c,v in ystats.items():var_rows.append({'Airline':a,'Null model':model,'Kind':'AUC outcome','Variable':' | '.join(c)}|v)
        X=sg[PREDICTORS].to_numpy(float);Y=wide.to_numpy(float);xr=normalized_ranks(X);yr=normalized_ranks(Y)
        rho=xr.T@yr;valid=np.array([xstats[p]['Status']=='usable' for p in PREDICTORS])[:,None]&np.array([ystats[c]['Status']=='usable' for c in wide])[None,:]
        rng=rng_for(9,ai,mi,0);extreme=np.zeros(rho.shape,dtype=int)
        for offset in range(0,NPERM,200):
            perms=np.array([rng.permutation(len(X)) for _ in range(min(200,NPERM-offset))])
            simulated=np.transpose(xr[perms],(0,2,1))@yr
            extreme+=(np.abs(simulated)>=np.abs(rho)[None,:,:]-1e-14).sum(axis=0)
        pvalues=(1+extreme)/(NPERM+1)
        boot=np.full((NBOOT,*rho.shape),np.nan);rng=rng_for(9,ai,mi,1)
        for b in range(NBOOT):
            sample=rng.integers(0,len(X),len(X));xx=X[sample];yy=Y[sample]
            rr=normalized_ranks(xx).T@normalized_ranks(yy)
            use=(np.ptp(np.round(xx,12),axis=0)>0)[:,None]&(np.ptp(np.round(yy,12),axis=0)>0)[None,:]
            boot[b][use]=rr[use]
        for pi,p in enumerate(PREDICTORS):
            for ci,c in enumerate(wide.columns):
                values=boot[:,pi,ci];values=values[np.isfinite(values)]
                ok=valid[pi,ci];ci_ok=ok and len(values)>=900
                lo,hi=np.percentile(values,[2.5,97.5]) if ci_ok else (np.nan,np.nan)
                assoc_rows.append({'Airline':a,'Null model':model,'Predictor':p,'Strategy':c[0],'Metric':c[1],'Window':c[2],'Networks':len(X),'Predictor status':xstats[p]['Status'],'Outcome status':ystats[c]['Status'],'Spearman rho':rho[pi,ci] if ok else np.nan,'Permutation p':pvalues[pi,ci] if ok else np.nan,'Pointwise CI low':lo,'Pointwise CI high':hi,'Valid bootstrap resamples':len(values),'Permutations':NPERM})
        redundancy=normalized_ranks(X).T@normalized_ranks(X)
        for i,p in enumerate(PREDICTORS):
            for j,q in enumerate(PREDICTORS):red_rows.append({'Airline':a,'Null model':model,'Predictor 1':p,'Predictor 2':q,'Spearman rho':redundancy[i,j] if xstats[p]['Unique']>1 and xstats[q]['Unique']>1 else np.nan})
        print(a,model,'associations complete',flush=True)
associations=pd.DataFrame(assoc_rows);associations['BY q']=np.nan
for model in ['reciprocal','directed']:
    mask=(associations['Null model']==model)&associations['Permutation p'].notna()
    associations.loc[mask,'BY q']=stats.false_discovery_control(associations.loc[mask,'Permutation p'].to_numpy(),method='by')
save(associations,'09_auc_associations.csv');save(pd.DataFrame(var_rows),'09_variability.csv');save(pd.DataFrame(red_rows),'09_predictor_redundancy.csv')
check('Declared AUC comparison family size',len(associations)==7*3*3*2*3*2)
display(associations[(associations['Null model']=='reciprocal')&(associations['BY q']<.05)].sort_values('BY q').head(20))

AA reciprocal associations complete
AA directed associations complete
DL reciprocal associations complete
DL directed associations complete
WN reciprocal associations complete
WN directed associations complete


,Airline,Null model,Predictor,Strategy,Metric,Window,Networks,Predictor status,Outcome status,Spearman rho,Permutation p,Pointwise CI low,Pointwise CI high,Valid bootstrap resamples,Permutations,BY q
108,AA,reciprocal,Degree assortativity,Betweenness,Efficiency retention,Early AUC,100,usable,usable,0.613233,0.00005,0.463642,0.733842,1000,19999,0.004245
109,AA,reciprocal,Degree assortativity,Betweenness,Efficiency retention,Full AUC,100,usable,usable,0.690557,0.00005,0.561363,0.780949,1000,19999,0.004245
110,AA,reciprocal,Degree assortativity,Betweenness,Reachability retention,Early AUC,100,usable,usable,0.630417,0.00005,0.484538,0.746365,1000,19999,0.004245
111,AA,reciprocal,Degree assortativity,Betweenness,Reachability retention,Full AUC,100,usable,usable,0.713660,0.00005,0.596718,0.795923,1000,19999,0.004245
112,AA,reciprocal,Degree assortativity,Betweenness,Largest SCC retention,Early AUC,100,usable,usable,0.654404,0.00005,0.515374,0.765693,1000,19999,0.004245
113,AA,reciprocal,Degree assortativity,Betweenness,Largest SCC retention,Full AUC,100,usable,usable,0.555805,0.00005,0.413912,0.674403,1000,19999,0.004245
114,AA,reciprocal,Degree assortativity,Degree,Efficiency retention,Early AUC,100,usable,usable,0.524166,0.00005,0.366334,0.649002,1000,19999,0.004245
115,AA,reciprocal,Degree assortativity,Degree,Efficiency retention,Full AUC,100,usable,usable,0.594361,0.00005,0.444511,0.702405,1000,19999,0.004245
116,AA,reciprocal,Degree assortativity,Degree,Reachability retention,Early AUC,100,usable,usable,0.539497,0.00005,0.383938,0.665441,1000,19999,0.004245
117,AA,reciprocal,Degree assortativity,Degree,Reachability retention,Full AUC,100,usable,usable,0.597746,0.00005,0.447081,0.702554,1000,19999,0.004245


## Focused visualizations

Keep the original four BY-adjusted heatmaps, a compact early-assortativity scatterplot,
and the reciprocal predictor-redundancy matrix. Remove the repetitive 7-by-3 scatter catalogs.
The heatmaps summarize a full exploratory family; a highlighted cell is not a new
confirmatory test. Observed graphs are *not* predicted from the within-null correlations.

In [5]:
from matplotlib.ticker import MaxNLocator

def association_heatmap(model, window):
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, airline in zip(axes, AIRLINES):
        sub = associations[(associations.Airline == airline) &
                           (associations['Null model'] == model) &
                           (associations.Window == window)]
        columns = [(s, m) for s in STRATEGIES for m in RETAIN]
        values = np.full((7, 9), np.nan)
        q = np.ones((7, 9))
        for i, predictor in enumerate(PREDICTORS):
            for j, (strategy, metric) in enumerate(columns):
                row = sub[(sub.Predictor == predictor) &
                          (sub.Strategy == strategy) &
                          (sub.Metric == metric)].iloc[0]
                values[i, j] = row['Spearman rho']
                q[i, j] = row['BY q']
        im = ax.imshow(values, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        for i, j in zip(*np.where(q < .05)):
            ax.text(j, i, '*', ha='center', va='center', color='black', fontsize=13)
        ax.set_xticks(range(9),
                      [s.split(' (')[0][:3] + ' ' + m.split()[0][:3] for s, m in columns],
                      rotation=60, ha='right', fontsize=8)
        ax.set_yticks(range(7), PREDICTORS if airline == 'AA' else [''] * 7, fontsize=9)
        ax.set_title(airline)
    fig.suptitle(f'{model}: {window} Spearman associations (* BY q<0.05; blank = excluded)')
    fig.subplots_adjust(left=.15, right=.86, bottom=.22, top=.86, wspace=.15)
    fig.colorbar(im, cax=fig.add_axes([.9, .27, .015, .5]), label='Spearman rho')
    figure('09_' + model + '_' + window.split()[0].lower() + '_associations')

for model in ['reciprocal', 'directed']:
    for window in ['Early AUC', 'Full AUC']:
        association_heatmap(model, window)

# Early-degree-targeted efficiency AUC is the most interpretable selected pattern.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, airline in zip(axes, AIRLINES):
    sub = dataset[(dataset.Airline == airline) &
                  (dataset['Network set'] == 'reciprocal') &
                  (dataset.Strategy == 'Degree') &
                  (dataset.Metric == 'Efficiency retention') &
                  (dataset.Window == 'Early AUC')]
    ax.scatter(sub['Degree assortativity'], sub['Normalized AUC'], s=22, alpha=.65)
    row = associations[(associations.Airline == airline) &
                       (associations['Null model'] == 'reciprocal') &
                       (associations.Predictor == 'Degree assortativity') &
                       (associations.Strategy == 'Degree') &
                       (associations.Metric == 'Efficiency retention') &
                       (associations.Window == 'Early AUC')].iloc[0]
    ax.set_title(f"{airline}: rho={row['Spearman rho']:.3f}, BY q={row['BY q']:.3g}")
    ax.set_xlabel('Degree assortativity')
    ax.grid(alpha=.2)
axes[0].set_ylabel('Normalized early resilience AUC')
fig.suptitle('Within reciprocal null ensembles; correlations are not interventions')
fig.tight_layout()
figure('09_reciprocal_assortativity_early_scatter')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
red = pd.DataFrame(red_rows)
for ax, airline in zip(axes, AIRLINES):
    sub = red[(red.Airline == airline) & (red['Null model'] == 'reciprocal')].pivot(
        index='Predictor 1', columns='Predictor 2', values='Spearman rho').reindex(
        index=PREDICTORS, columns=PREDICTORS)
    im = ax.imshow(sub, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(7), PREDICTORS, rotation=75, ha='right', fontsize=7)
    ax.set_yticks(range(7), PREDICTORS if airline == 'AA' else [''] * 7, fontsize=7)
    ax.set_title(airline)
fig.suptitle('Reciprocal ensembles: structural predictor redundancy')
fig.tight_layout(rect=[0, 0, .9, .93])
fig.colorbar(im, cax=fig.add_axes([.93, .30, .012, .45]), label='Spearman rho')
figure('09_predictor_redundancy')
finish()

09: 954 checks passed
